# Notebook Overview — Basic Testing

## Purpose

This notebook establishes the **baseline performance** of the proposed Digital Image Processing (DIP) feature-based approach for AI-generated image detection.

The goal is to evaluate how well the engineered feature vector performs using a **standard classifier configuration**, without extensive hyperparameter tuning.

This provides a reference point for later improvement and analysis.

---

## Inputs

The notebook expects the following:

* Training feature vectors from the GitHub repository:

  * `metadata/vectors/train_feature_vectors.csv`

* Test feature vectors from the GitHub repository:

  * `metadata/vectors/test_feature_vectors.csv`

* Project configuration file:

  * `src/project_config.py`

---

## Assumptions

* Each feature vector CSV contains the columns:

  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* The remaining columns represent the engineered DIP feature vector (**25 features**)

* The feature vector data has already been created by earlier notebooks in the pipeline

* The training and test sets are already defined and must remain separate

* This notebook uses the training set for model fitting and the test set for independent evaluation

* This notebook focuses on **baseline testing only** and does not perform extensive tuning

* All paths and file locations are controlled through `project_config.py`

* All outputs are written to **local runtime storage only**

* No files are written back to Google Drive

---

## What This Notebook Does

### Cell 1 — Environment Setup

* Clone the GitHub repository into the local runtime (if needed)
* Import project configuration (`project_config.py`)
* Verify required input files are available using config-defined paths

---

### Cell 2 — Load Feature Vector Data

* Load the training and test feature vector CSV files
* Display dataset shapes
* Verify expected metadata columns are present
* Validate subset labels, class labels, and data integrity

---

### Cell 3 — Prepare Features and Labels

* Separate metadata columns from feature columns using config definitions
* Create feature matrices (`X_train`, `X_test`)
* Create label vectors (`y_train`, `y_test`)
* Verify feature dimensionality (**25 features**) and class labels

---

### Cell 4 — Normalize Features

* Fit a scaler using the training feature matrix only
* Transform both training and test feature matrices
* Preserve strict train/test separation during normalization
* Save the fitted scaler using a config-defined path

---

### Cell 5 — Define Baseline Classifier

* Define the baseline classifier configuration
* Use the Multi-Layer Perceptron (MLP) as the primary baseline model
* Use a standard architecture `(64, 32)` with default learning settings

---

### Cell 6 — Train Baseline Model

* Fit the baseline classifier using the normalized training data
* Record training time and convergence behavior
* Capture convergence warnings if present

---

### Cell 7 — Evaluate on Test Set

* Generate predictions on the independent test set
* Compute evaluation metrics:

  * Accuracy  
  * Precision  
  * Recall  
  * F1-score  
  * ROC AUC  

* Compute confusion matrix and classification report

---

### Cell 8 — Display Results

* Present baseline metrics in a compact summary table
* Optionally display confusion matrix and ROC curve visualizations (controlled by `VERBOSE`)

---

### Cell 9 — Save Results

* Save baseline results using config-defined output paths:

  * `metadata/results/basic_testing_results.csv`
  * `metadata/results/baseline_model_config.json`

* Save scaler:

  * `metadata/models/scaler.pkl`

* Verify output file creation

---

### Cell 10 — Final Summary

* Display final baseline testing summary
* Report model configuration and performance metrics
* Provide guidance for the next notebook in the pipeline

---

## Outputs

The following outputs are generated in the **local runtime**:

* Baseline classification metrics  
* Confusion matrix  
* ROC curve  

Saved output files (config-controlled paths):

* `metadata/results/basic_testing_results.csv`
* `metadata/results/baseline_model_config.json`
* `metadata/models/scaler.pkl`

These files are not written back to Google Drive and will be lost when the runtime ends unless explicitly preserved.

---

## Expected Data Scope

* Training feature vectors: **14,400 rows**  
* Test feature vectors: **3,600 rows**

Each row corresponds to one image and includes metadata plus the engineered **25-dimensional DIP feature vector**.

---

## Notes

* This notebook evaluates the **baseline performance** of the DIP feature-based method  
* The primary goal is to determine how well the engineered features perform using a standard classifier setup  
* The test set is used only for independent evaluation and is not used during training  
* The baseline model may reach the maximum iteration limit without full convergence; this is expected and does not prevent valid evaluation  
* This notebook establishes the reference point for later improvement in the fine-tuning stage  
* The Multi-Layer Perceptron (MLP) is the primary baseline model used in this notebook  
* Optional displays are controlled via the `VERBOSE` flag  

---

**Next step:**  
After baseline results are established, proceed to **11_Basic_Fine-Tuning.ipynb** to explore controlled improvement through limited hyperparameter tuning.

In [ ]:
# ============================================
# Cell 1: Startup (Environment + Verification)
# ============================================

import os
import sys
from pathlib import Path

import pandas as pd

from sklearn.preprocessing import StandardScaler
import joblib

# ------------------------------------------------------------
# Notebook display control
# ------------------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# ------------------------------------------------------------
# Suppress warnings for cleaner notebook output
# ------------------------------------------------------------
if not VERBOSE:
    warnings.filterwarnings("ignore")

# -------------------------------------------------
# Clone repo into Colab runtime if not already present
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import *

# -------------------------------------------------
# Convert config paths to Path objects
# -------------------------------------------------
train_feature_path = Path(TRAIN_FEATURE_VECTOR_PATH)
test_feature_path = Path(TEST_FEATURE_VECTOR_PATH)

results_dir = Path(RESULTS_METADATA_DIR)
models_dir = Path(MODELS_METADATA_DIR)

basic_testing_results_path = Path(BASIC_TESTING_RESULTS_PATH)
baseline_model_config_path = Path(BASELINE_MODEL_CONFIG_PATH)
scaler_path = Path(SCALER_PATH)

# -------------------------------------------------
# Ensure output directories exist
# -------------------------------------------------
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required input files...\n")

required_files = {
    "Training feature vectors": train_feature_path,
    "Test feature vectors": test_feature_path,
}

missing_files = [
    f"{description}: {path}"
    for description, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required input file(s):\n" + "\n".join(missing_files)
    )

print("All required input files are present.")

if VERBOSE:
    print(f"\nTraining features: {train_feature_path}")
    print(f"Test features:     {test_feature_path}")
    print(f"Results output:    {basic_testing_results_path}")
    print(f"Model config:      {baseline_model_config_path}")
    print(f"Scaler output:     {scaler_path}")

print("\nStartup complete.")



In [ ]:
# ============================================
# Cell 2: Load Feature Vector Data
# ============================================

# -------------------------------------------------
# Load training and test feature vectors
# -------------------------------------------------
train_df = pd.read_csv(train_feature_path)
test_df  = pd.read_csv(test_feature_path)

print("Feature vector files loaded successfully.\n")

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape:     {test_df.shape}")

# -------------------------------------------------
# Verify required columns
# -------------------------------------------------
required_columns = ["filename", "class_label", "source_dataset", "subset"]

missing_train = [col for col in required_columns if col not in train_df.columns]
missing_test  = [col for col in required_columns if col not in test_df.columns]

if missing_train:
    raise ValueError(f"Training file missing required columns: {missing_train}")

if missing_test:
    raise ValueError(f"Test file missing required columns: {missing_test}")

print("\nRequired metadata columns verified.")

# -------------------------------------------------
# Verify subset labels
# -------------------------------------------------
train_subsets = train_df["subset"].unique()
test_subsets  = test_df["subset"].unique()

if list(train_subsets) != ["train"]:
    raise ValueError(f"Unexpected subset values in training data: {train_subsets}")

if list(test_subsets) != ["test"]:
    raise ValueError(f"Unexpected subset values in test data: {test_subsets}")

print("Subset labels verified.")

# -------------------------------------------------
# Check for missing values
# -------------------------------------------------
if train_df.isnull().values.any():
    raise ValueError("Training data contains missing values.")

if test_df.isnull().values.any():
    raise ValueError("Test data contains missing values.")

print("No missing values detected.")

# -------------------------------------------------
# Display class distribution
# -------------------------------------------------
print("\nTraining class distribution:")
print(train_df["class_label"].value_counts())

print("\nTest class distribution:")
print(test_df["class_label"].value_counts())

# -------------------------------------------------
# Optional display of sample rows
# -------------------------------------------------
if VERBOSE:
    print("\nSample training rows:")
    display(train_df.head())

    print("\nSample test rows:")
    display(test_df.head())



In [ ]:
# ============================================
# Cell 3: Prepare Features and Labels
# ============================================

# -------------------------------------------------
# Use config-defined metadata columns
# -------------------------------------------------
metadata_columns = METADATA_COLUMNS

feature_columns = [col for col in train_df.columns if col not in metadata_columns]

print(f"Number of feature columns: {len(feature_columns)}")
print(f"Expected number of features: {NUM_FEATURES}")

# -------------------------------------------------
# Validate feature count
# -------------------------------------------------
if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Feature count mismatch. Expected {NUM_FEATURES}, found {len(feature_columns)}"
    )

# -------------------------------------------------
# Create feature matrices
# -------------------------------------------------
X_train = train_df[feature_columns].values
X_test  = test_df[feature_columns].values

# -------------------------------------------------
# Create label vectors
# -------------------------------------------------
y_train = train_df["class_label"].values
y_test  = test_df["class_label"].values

# -------------------------------------------------
# Verify feature consistency
# -------------------------------------------------
if X_train.shape[1] != X_test.shape[1]:
    raise ValueError(
        f"Feature dimension mismatch: train={X_train.shape[1]}, test={X_test.shape[1]}"
    )

print("\nFeature matrices created.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

# -------------------------------------------------
# Verify class labels
# -------------------------------------------------
unique_labels = sorted(set(y_train) | set(y_test))

print(f"\nUnique class labels: {unique_labels}")

if set(unique_labels) != set(VALID_LABELS):
    raise ValueError(
        f"Unexpected class labels. Expected {VALID_LABELS}, found {unique_labels}"
    )

print("Class labels verified.")

# -------------------------------------------------
# Optional display
# -------------------------------------------------
if VERBOSE:
    print("\nSample feature vector (first row):")
    print(X_train[0])



In [ ]:
# ============================================
# Cell 4: Normalize Features
# ============================================

# -------------------------------------------------
# Use config-defined scaler path
# -------------------------------------------------
scaler_file = scaler_path

# -------------------------------------------------
# Fit scaler on training data only
# -------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Feature normalization complete.\n")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

# -------------------------------------------------
# Save scaler to local runtime
# -------------------------------------------------
joblib.dump(scaler, scaler_file)

if not scaler_file.exists():
    raise FileNotFoundError(f"Scaler file was not created: {scaler_file}")

print(f"Scaler saved to: {scaler_file}")

# -------------------------------------------------
# Sanity check on normalized training data
# -------------------------------------------------
train_mean = X_train_scaled.mean(axis=0)
train_std = X_train_scaled.std(axis=0)

if VERBOSE:
    print("\nNormalization sanity check:")
    print(f"Mean of first 5 scaled features: {train_mean[:5]}")
    print(f"Std of first 5 scaled features:  {train_std[:5]}")



In [ ]:
# ============================================
# Cell 5: Define Baseline Classifier
# ============================================

from sklearn.neural_network import MLPClassifier

# -------------------------------------------------
# Define baseline MLP configuration
# -------------------------------------------------
baseline_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=300,
    random_state=RANDOM_SEED,
    early_stopping=False,
)

print("Baseline classifier defined.\n")

print("Model type:")
print(type(baseline_mlp).__name__)

# -------------------------------------------------
# Optional detailed configuration display
# -------------------------------------------------
if VERBOSE:
    print("\nBaseline MLP configuration:")
    print(f"hidden_layer_sizes : {baseline_mlp.hidden_layer_sizes}")
    print(f"activation         : {baseline_mlp.activation}")
    print(f"solver             : {baseline_mlp.solver}")
    print(f"alpha              : {baseline_mlp.alpha}")
    print(f"batch_size         : {baseline_mlp.batch_size}")
    print(f"learning_rate_init : {baseline_mlp.learning_rate_init}")
    print(f"max_iter           : {baseline_mlp.max_iter}")
    print(f"random_state       : {baseline_mlp.random_state}")
    print(f"early_stopping     : {baseline_mlp.early_stopping}")



In [ ]:
# ============================================
# Cell 6: Train Baseline Model
# ============================================

import time
import warnings
from sklearn.exceptions import ConvergenceWarning

# -------------------------------------------------
# Train baseline model
# -------------------------------------------------
print("Training baseline MLP model...\n")

start_time = time.time()

with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    baseline_mlp.fit(X_train_scaled, y_train)

end_time = time.time()
training_time = end_time - start_time

print("Training complete.\n")
print(f"Training time: {training_time:.2f} seconds")

# -------------------------------------------------
# Report convergence information
# -------------------------------------------------
if VERBOSE:
    print(f"\nNumber of iterations completed: {baseline_mlp.n_iter_}")
    print(f"Final training loss: {baseline_mlp.loss_:.6f}")

convergence_warnings = [
    w for w in caught_warnings
    if issubclass(w.category, ConvergenceWarning)
]

if convergence_warnings:
    print("\nWARNING: Convergence warning detected.")
    print("The optimizer did not fully converge within the allowed iterations.")
else:
    print("\nModel converged without warnings.")



In [ ]:
# ============================================
# Cell 7: Evaluate on Test Set
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# -------------------------------------------------
# Generate predictions
# -------------------------------------------------
y_pred = baseline_mlp.predict(X_test_scaled)

# Find index of AI class
ai_index = list(baseline_mlp.classes_).index(AI_LABEL)

y_prob = baseline_mlp.predict_proba(X_test_scaled)[:, ai_index]

# -------------------------------------------------
# Convert labels to binary for ROC AUC
# -------------------------------------------------
y_test_binary = (y_test == AI_LABEL).astype(int)

# -------------------------------------------------
# Compute evaluation metrics
# -------------------------------------------------
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=AI_LABEL)
recall = recall_score(y_test, y_pred, pos_label=AI_LABEL)
f1 = f1_score(y_test, y_pred, pos_label=AI_LABEL)
roc_auc = roc_auc_score(y_test_binary, y_prob)

cm = confusion_matrix(y_test, y_pred, labels=[REAL_LABEL, AI_LABEL])

# -------------------------------------------------
# Display results
# -------------------------------------------------
print("Baseline test-set evaluation complete.\n")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix (rows=true, cols=pred):")
print(cm)

if VERBOSE:
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))



In [ ]:
# ============================================
# Cell 8: Display Results
# ============================================

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
import pandas as pd

# -------------------------------------------------
# Create compact results summary table
# -------------------------------------------------
results_summary_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

print("Baseline performance summary:")
display(results_summary_df)

# -------------------------------------------------
# Optional visual displays
# -------------------------------------------------
if VERBOSE:
    # -------------------------------------------------
    # Confusion matrix display
    # -------------------------------------------------
    disp_cm = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[REAL_LABEL, AI_LABEL]
    )

    plt.figure(figsize=(6, 6))
    disp_cm.plot(cmap="Blues", values_format="d")
    plt.title("Baseline MLP Confusion Matrix")
    plt.grid(False)
    plt.show()

    # -------------------------------------------------
    # ROC curve display
    # -------------------------------------------------
    RocCurveDisplay.from_predictions(
        y_test_binary,
        y_prob,
        name="Baseline MLP"
    )

    plt.title("Baseline MLP ROC Curve")
    plt.grid(True)
    plt.show()



In [ ]:
# ============================================
# Cell 9: Save Results
# ============================================

import pandas as pd
import json

# -------------------------------------------------
# Create results dictionary
# -------------------------------------------------
results_dict = {
    "model": "MLP (64, 32)",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "roc_auc": roc_auc,
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "num_features": X_train.shape[1],
    "training_iterations": baseline_mlp.n_iter_,
    "final_loss": baseline_mlp.loss_,
}

# -------------------------------------------------
# Save results to CSV (config path)
# -------------------------------------------------
results_df = pd.DataFrame([results_dict])
results_df.to_csv(basic_testing_results_path, index=False)

# -------------------------------------------------
# Verify file creation
# -------------------------------------------------
if not basic_testing_results_path.exists():
    raise FileNotFoundError(
        f"Results file was not created: {basic_testing_results_path}"
    )

print(f"Results saved to: {basic_testing_results_path}")

# -------------------------------------------------
# Save model configuration (config path)
# -------------------------------------------------
model_config = {
    "model_type": "MLPClassifier",
    "hidden_layer_sizes": baseline_mlp.hidden_layer_sizes,
    "activation": baseline_mlp.activation,
    "solver": baseline_mlp.solver,
    "alpha": baseline_mlp.alpha,
    "batch_size": baseline_mlp.batch_size,
    "learning_rate_init": baseline_mlp.learning_rate_init,
    "max_iter": baseline_mlp.max_iter,
    "random_state": baseline_mlp.random_state,
}

with open(baseline_model_config_path, "w") as f:
    json.dump(model_config, f, indent=4)

print(f"Model configuration saved to: {baseline_model_config_path}")

print("\nCell complete.")



In [ ]:
# ============================================
# Cell 10: Final Summary and Next Step
# ============================================

from IPython.display import display, HTML

print("Basic baseline testing completed successfully.\n")

print("Baseline Model Summary:")
print(f"Model type         : {type(baseline_mlp).__name__}")
print(f"Architecture       : {baseline_mlp.hidden_layer_sizes}")
print(f"Training samples   : {len(X_train)}")
print(f"Test samples       : {len(X_test)}")
print(f"Feature count      : {X_train.shape[1]}")
print(f"Training iterations: {baseline_mlp.n_iter_}")
print(f"Final training loss: {baseline_mlp.loss_:.6f}")

print("\nBaseline Test Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix Summary:")
print(f"Real ({REAL_LABEL}): {cm[0,0]} correct, {cm[0,1]} misclassified")
print(f"AI   ({AI_LABEL}): {cm[1,1]} correct, {cm[1,0]} misclassified")
print()

# -------------------------------------------------
# Next step message
# -------------------------------------------------
message = """
<b>Baseline Testing Complete:</b><br>
The baseline MLP model has been trained and evaluated on the independent test set.<br><br>
Proceed to <b>11_Basic_Fine-Tuning.ipynb</b> to explore controlled improvement through limited hyperparameter tuning.
"""

display(HTML(f"""
<div style="
    padding: 15px;
    border: 2px solid #4CAF50;
    background-color: #E8F5E9;
    border-radius: 8px;
    font-size: 16px;
">
{message}
</div>
"""))

